In [ ]:

import pandas as pd
from google.colab import files

uploaded = files.upload()

file_name = next(
    name for name in uploaded if name.endswith(".csv")
)

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())
print(df.head())

Saving english_hinglish_sample_dataset.csv to english_hinglish_sample_dataset (1).csv
Dataset shape: (152, 2)
Column names: ['english', 'hinglish']
          english        hinglish
0           Hello         Namaste
1              Hi              Hi
2    Good morning    Good morning
3  Good afternoon  Good afternoon
4    Good evening    Good evening


In [ ]:

import re
import numpy as np

# CHANGE these to your actual CSV column names
SOURCE_COL = "english"
TARGET_COL = "hinglish"

data = df[[SOURCE_COL, TARGET_COL]].dropna().copy()

def clean_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

data[SOURCE_COL] = data[SOURCE_COL].apply(clean_text)
data[TARGET_COL] = data[TARGET_COL].apply(clean_text)

# Remove empty sentence pairs
data = data[
    (data[SOURCE_COL] != "") &
    (data[TARGET_COL] != "")
]

# Add start and end markers for the decoder
data[TARGET_COL] = (
    "startseq " + data[TARGET_COL] + " endseq"
)

print(data.head())
print("Total sentence pairs:", len(data))


          english                        hinglish
0           hello         startseq namaste endseq
1              hi              startseq hi endseq
2    good morning    startseq good morning endseq
3  good afternoon  startseq good afternoon endseq
4    good evening    startseq good evening endseq
Total sentence pairs: 152


In [ ]:

import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

MAX_VOCAB = 15000
MAX_LEN = 30

source_vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN
)

target_vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN
)

source_vectorizer.adapt(
    data[SOURCE_COL].values
)

target_vectorizer.adapt(
    data[TARGET_COL].values
)

X = source_vectorizer(
    data[SOURCE_COL].values
).numpy()

Y = target_vectorizer(
    data[TARGET_COL].values
).numpy()

source_vocab_size = (
    len(source_vectorizer.get_vocabulary())
)

target_vocab_size = (
    len(target_vectorizer.get_vocabulary())
)

print("Source vocabulary:", source_vocab_size)
print("Target vocabulary:", target_vocab_size)
print("English sequence shape:", X.shape)
print("Target sequence shape:", Y.shape)


Source vocabulary: 226
Target vocabulary: 259
English sequence shape: (152, 30)
Target sequence shape: (152, 30)


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 121
Testing samples: 31


In [ ]:

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense
)

EMBED_DIM = 128
LATENT_DIM = 256

# Encoder
encoder_inputs = Input(
    shape=(MAX_LEN,), name="encoder_inputs"
)

encoder_embedding = Embedding(
    source_vocab_size, EMBED_DIM, mask_zero=True
)

encoder_embedded = encoder_embedding(encoder_inputs)

encoder_lstm = LSTM(
    LATENT_DIM, return_state=True, name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_embedded)

encoder_states = [state_h, state_c]

# Decoder: teacher forcing during training
decoder_inputs = Input(
    shape=(MAX_LEN - 1,), name="decoder_inputs"
)

decoder_embedding = Embedding(
    target_vocab_size, EMBED_DIM, mask_zero=True
)

decoder_embedded = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedded,
    initial_state=encoder_states
)

decoder_dense = Dense(
    target_vocab_size, activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 29)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 30, 128)   │     28,928 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 30)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 29, 128)   │     33,152 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 29, 256), │    394,240 │ embedding_1[0][0… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 29, 259)   │     66,563 │ decoder_lstm[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 917,123 (3.50 MB)

 Trainable params: 917,123 (3.50 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

# Decoder input: remove the last target token
decoder_train_input = Y_train[:, :-1]

# Expected output: remove the first target token
decoder_train_target = Y_train[:, 1:]

decoder_test_input = Y_test[:, :-1]
decoder_test_target = Y_test[:, 1:]

# Ignore padding tokens (ID 0) in loss and accuracy
train_weights = (
    decoder_train_target != 0
).astype("float32")

test_weights = (
    decoder_test_target != 0
).astype("float32")

history = model.fit(
    [X_train, decoder_train_input],
    decoder_train_target,
    sample_weight=train_weights,
    validation_data=(
        [X_test, decoder_test_input],
        decoder_test_target,
        test_weights
    ),
    epochs=10,
    batch_size=64
)


Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 460ms/step - loss: 4.5864 - sparse_categorical_accuracy: 0.0878 - val_loss: 4.5924 - val_sparse_categorical_accuracy: 0.1713
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 4.5700 - sparse_categorical_accuracy: 0.1770 - val_loss: 4.5750 - val_sparse_categorical_accuracy: 0.1713
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - loss: 4.5460 - sparse_categorical_accuracy: 0.1741 - val_loss: 4.5428 - val_sparse_categorical_accuracy: 0.1713
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - loss: 4.5018 - sparse_categorical_accuracy: 0.1741 - val_loss: 4.4770 - val_sparse_categorical_accuracy: 0.1713
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - loss: 4.4060 - sparse_categorical_accuracy: 0.1741 - val_loss: 4.3336 - val_sparse_categorical_accuracy: 0.1713
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 4.2067 - sparse_categorical_accuracy: 0.1741 - val_loss: 4.0321 - val_sparse_categorical_accuracy: 0.1713
Epoch 7/10
2/2 ━━━━━

In [ ]:

loss, accuracy = model.evaluate(
    [X_test, decoder_test_input],
    decoder_test_target,
    sample_weight=test_weights,
    verbose=1
)

print("Test Loss:", loss)
print("Test Token Accuracy:", accuracy)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 3.6738 - sparse_categorical_accuracy: 0.1713
Test Loss: 3.6738436222076416
Test Token Accuracy: 0.1712707132101059


In [ ]:

# Encoder inference model
encoder_model = Model(
    encoder_inputs, encoder_states
)

# Decoder inference inputs: one token at a time
decoder_state_h = Input(shape=(LATENT_DIM,))
decoder_state_c = Input(shape=(LATENT_DIM,))

decoder_step_input = Input(shape=(1,))

step_embedded = decoder_embedding(decoder_step_input)

step_output, next_h, next_c = decoder_lstm(
    step_embedded,
    initial_state=[decoder_state_h, decoder_state_c]
)

step_output = decoder_dense(step_output)

decoder_model = Model(
    [decoder_step_input, decoder_state_h, decoder_state_c],
    [step_output, next_h, next_c]
)

# Word ID lookup
target_vocab = target_vectorizer.get_vocabulary()
target_word_to_id = {
    word: i for i, word in enumerate(target_vocab)
}

start_id = target_word_to_id.get("startseq")
end_id = target_word_to_id.get("endseq")

def translate(sentence):
    sentence = clean_text(sentence)

    input_seq = source_vectorizer(
        tf.constant([sentence])
    ).numpy()

    states = encoder_model.predict(
        input_seq, verbose=0
    )

    token_id = start_id
    result = []

    for _ in range(MAX_LEN):
        token = np.array([[token_id]])

        output, h, c = decoder_model.predict(
            [token, states[0], states[1]],
            verbose=0
        )

        token_id = int(np.argmax(output[0, -1, :]))
        states = [h, c]

        if token_id == end_id or token_id == 0:
            break

        word = target_vocab[token_id]

        if word not in ("[UNK]", "startseq"):
            result.append(word)

    return " ".join(result)


print("Translation:", translate("How are you?"))


Translation: 


In [ ]:

!pip -q install nltk


In [ ]:

from nltk.translate.bleu_score import (
    corpus_bleu, SmoothingFunction
)

references = []
hypotheses = []

# Evaluate up to 100 held-out sentence pairs
limit = min(100, len(X_test))

for i in range(limit):
    # Recover the original source sentence
    source_sentence = source_vectorizer(
        tf.constant([data[SOURCE_COL].iloc[0]])
    )  # Demonstration only; use the held-out rows below


In [ ]:

# Split the original sentence pairs using the same split
train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

references = []
hypotheses = []

for _, row in test_df.head(100).iterrows():
    predicted = translate(row[SOURCE_COL])

    # Remove the start/end markers from the reference
    actual = row[TARGET_COL].replace(
        "startseq ", ""
    ).replace(" endseq", "")

    references.append([actual.split()])
    hypotheses.append(predicted.split())

bleu = corpus_bleu(
    references,
    hypotheses,
    smoothing_function=SmoothingFunction().method1
)

print("BLEU Score:", round(bleu, 4))
print("BLEU Percentage:", round(bleu * 100, 2), "%")


BLEU Score: 0
BLEU Percentage: 0 %


In [ ]:

test_sentences = [
    "How are you?",
    "What is your name?",
    "I am going to college.",
    "Please help me.",
    "Today is a beautiful day."
]

for sentence in test_sentences:
    print("English:", sentence)
    print("Hinglish:", translate(sentence))
    print("-" * 40)


English: How are you?
Hinglish: 
----------------------------------------
English: What is your name?
Hinglish: 
----------------------------------------
English: I am going to college.
Hinglish: 
----------------------------------------
English: Please help me.
Hinglish: 
----------------------------------------
English: Today is a beautiful day.
Hinglish: 
----------------------------------------


In [ ]:

model.save("/content/english_hinglish_model.keras")

print("Model saved successfully!")


Model saved successfully!


In [ ]:

from google.colab import files

files.download("/content/english_hinglish_model.keras")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>